# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassaan-Raza/FlyRank-Intership/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Rule in plain words: flag content that is both declining and still has real
search demand behind it, that combination means there's something worth
reviewing, not just noise.

Signal A (staleness): [fill in verdict once you see the table, e.g. CONFIRMED
if decline_rate climbs with staleness tier]
Signal B (CTR vs position): [fill in verdict, e.g. CONFIRMED if avg_ctr drops
as position tier worsens]

Reason code this rule outputs: declining_with_demand

In [1]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

# Signal A: staleness (behind the refresh flags)
df['staleness_tier'] = pd.cut(
    df['days_since_last_update'],
    bins=[-1, 90, 180, 365, 100000],
    labels=['<90d', '90-180d', '180-365d', '365d+']
)
staleness_bucket = df.groupby('staleness_tier').agg(
    n=('content_id', 'count'),
    decline_rate=('trend_direction', lambda x: (x == 'down').mean())
).reset_index()
print("Signal A: staleness")
print(staleness_bucket)

# Signal B: CTR vs position (behind the CTR-fix logic)
df['position_tier'] = pd.cut(
    df['avg_position'],
    bins=[0, 3, 10, 20, 100],
    labels=['1-3', '4-10', '11-20', '21+']
)
ctr_bucket = df.groupby('position_tier').agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean')
).reset_index()
print("\nSignal B: CTR vs position")
print(ctr_bucket)

Signal A: staleness
  staleness_tier      n  decline_rate
0           <90d  20655      0.512031
1        90-180d   9171      0.611057
2       180-365d    169      0.467456
3          365d+      5      0.600000

Signal B: CTR vs position
  position_tier      n   avg_ctr
0           1-3   1141  2.714303
1          4-10  11842  0.651045
2         11-20   7273  0.323443
3           21+   8524  0.211705


/tmp/ipykernel_1945/3190042256.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_bucket = df.groupby('staleness_tier').agg(
/tmp/ipykernel_1945/3190042256.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_bucket = df.groupby('position_tier').agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
flagged = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)].copy()
flagged['score'] = flagged['impressions_90d']
flagged['reason_code'] = 'declining_with_demand'
flagged['action'] = 'review_for_refresh'

queue = flagged[['content_id', 'score', 'reason_code', 'action']].sort_values('score', ascending=False)

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(queue.head(20))
print(f"\nTotal flagged: {len(queue)}")

                 content_id   score            reason_code              action
6653   content_5fe46e04994d  517715  declining_with_demand  review_for_refresh
26844  content_8c19996aa890  509252  declining_with_demand  review_for_refresh
21819  content_4c36c775b818  463103  declining_with_demand  review_for_refresh
29879  content_1a9e894be2e2  416180  declining_with_demand  review_for_refresh
13537  content_2c2606c5d176  347399  declining_with_demand  review_for_refresh
26531  content_cb112fce36be  309910  declining_with_demand  review_for_refresh
21565  content_9532f197bbc8  309192  declining_with_demand  review_for_refresh
27478  content_008fb02c46cb  236803  declining_with_demand  review_for_refresh
23767  content_813e88069237  233561  declining_with_demand  review_for_refresh
26304  content_ff94c9b6b411  228566  declining_with_demand  review_for_refresh
15968  content_66b4046cc144  217415  declining_with_demand  review_for_refresh
10741  content_07e0b9af8b1a  214816  declining_with_

## 3. Top-20 review

1. content_5fe46e04994d - score 517,715 - Action: review_for_refresh. Why: highest impressions in the entire flagged set with a declining trend, maximum demand at stake. What would make it wrong: if this is a single seasonal spike (e.g. a news-driven page) rather than a sustained decline, the drop could be natural cooldown, not a real problem.

2. content_8c19996aa890 - score 509,252 - Action: review_for_refresh. Why: near-identical scale to #1, very high demand with a down trend. What would make it wrong: same seasonality risk as #1, also worth checking if it's cannibalized by a related page rather than genuinely declining.

3. content_4c36c775b818 - score 463,103 - Action: review_for_refresh. Why: still extremely high demand, comfortably above the next tier down. What would make it wrong: could be a page nearing natural end-of-relevance (e.g. an outdated product page) where refresh effort wouldn't pay off.

4. content_1a9e894be2e2 - score 416,180 - Action: review_for_refresh. Why: high impressions with confirmed decline signal. What would make it wrong: same general risk, no way to distinguish real decline from consolidation without checking sibling pages.

5. content_2c2606c5d176 - score 347,399 - Action: review_for_refresh. Why: still well above the median score, meaningful demand. What would make it wrong: this rule can't see whether traffic moved to a related URL on the same site.

6. content_cb112fce36be - score 309,910 - Action: review_for_refresh. Why: high demand, declining trend confirmed. What would make it wrong: possible SERP-level change (search page changed) rather than a content quality problem.

7. content_9532f197bbc8 - score 309,192 - Action: review_for_refresh. Why: near-identical score to #6, same reasoning applies. What would make it wrong: same SERP-change risk, position/CTR should be checked separately before committing review time.

8. content_008fb02c46cb - score 236,803 - Action: review_for_refresh. Why: noticeable drop-off from the top group but still far above median. What would make it wrong: could be legitimate seasonal demand shift rather than quality decline.

9. content_813e88069237 - score 233,561 - Action: review_for_refresh. Why: high demand, consistent with the rule's intent. What would make it wrong: same general caveats, no persistence check built into this simple rule.

10. content_ff94c9b6b411 - score 228,566 - Action: review_for_refresh. Why: still comfortably in the top tier of flagged pages. What would make it wrong: this rule has no minimum-persistence requirement, a short-lived dip could trigger the same flag as a real decline.

11. content_66b4046cc144 - score 217,415 - Action: review_for_refresh. Why: high demand, declining trend. What would make it wrong: same persistence gap as #10.

12. content_07e0b9af8b1a - score 214,816 - Action: review_for_refresh. Why: comparable scale to neighboring rows. What would make it wrong: no way to rule out noise at this rule's current level of detail.

13. content_cea79ef51519 - score 208,798 - Action: review_for_refresh. Why: still well above median, real demand. What would make it wrong: same general risks as above.

14. content_c8e9d6ab9013 - score 208,678 - Action: review_for_refresh. Why: near-tied with #13. What would make it wrong: same reasoning, no distinguishing signal at this tier.

15. content_bf7bff5d0756 - score 197,199 - Action: review_for_refresh. Why: consistent with the rule's target profile. What would make it wrong: same general caveats.

16. content_9463d30d5826 - score 192,478 - Action: review_for_refresh. Why: still solidly in the flagged demand range. What would make it wrong: same general caveats.

17. content_3d94572c3a35 - score 190,623 - Action: review_for_refresh.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

Weakest picks: none of the top 20 are individually weak by this rule's own logic, they're all extreme outliers (median score across all 13,152 flagged rows is 1,620, versus 174,000+ for the entire top 20). The actual weakness is structural, not row-specific: this rule has no persistence check and no consolidation/seasonality check, so a page that dropped sharply just once, or lost traffic to a sibling page, would score identically to a page in genuine sustained decline. Rows 6-7 and 13-14 are near-duplicate scores, worth spot-checking whether they're related content items that could be cannibalizing each other rather than two independent problems.

Leakage check: this rule uses only trend_direction and impressions_90d, both
current-window observed signals available at decision time. No FlyRank
product-computed score (health_score, priority_score, action_type) was used
as an input, and no future-window data was used, trend_direction reflects
only the current period, not a forecast.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.